# SOEA-Plus Protocol A — Full Re-Run (N=1000)
## GPT-4.1-mini | Gemini-2.5-Flash | Llama-3.3-70b

**What this notebook does:**
- Runs all 3 models on 1000 PubMedQA cases (Protocol A)
- Uses `max_tokens=300` (fixes the original truncation issue)
- Saves **full raw responses** for every case
- Produces confusion matrices, accuracy, parse failure rates
- Computes majority-class and random baselines
- Saves clean CSV ready for paper Table 1

**Estimated run time:** ~60-90 minutes total  
**Estimated cost:** GPT ~$0.30 | Gemini free | Llama ~$0.50

**Output files:**
- `protocol_a_results_gpt.csv`
- `protocol_a_results_gemini.csv`
- `protocol_a_results_llama.csv`
- `protocol_a_results_ALL.csv` (combined)
- `protocol_a_confusion_matrices.png`
- `protocol_a_summary.txt`


## Cell 1 — Install Packages

In [ ]:
import subprocess, sys
pkgs = ['openai', 'datasets', 'pandas', 'numpy',
        'matplotlib', 'seaborn', 'scipy', 'tqdm', 'requests']
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q'] + pkgs)
print('Done.')


## Cell 2 — API Keys & Config

In [ ]:
# ─── API KEYS ────────────────────────────────────────────────────────────
OPENAI_API_KEY  = os.environ.get("OPENAI_API_KEY", "")   # set env var before running
GEMINI_API_KEY  = os.environ.get("GEMINI_API_KEY", "")   # set env var before running
TOGETHER_API_KEY = os.environ.get("TOGETHER_API_KEY", "") # set env var before running
# ─────────────────────────────────────────────────────────────────────────

N_SAMPLES = 1000   # Full Protocol A
SEED      = 42
BATCH_SAVE = 50    # Save checkpoint every 50 cases

import os, re, time, warnings, json
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import requests
import openai
warnings.filterwarnings('ignore')

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)

print(f'Config ready. N={N_SAMPLES}, SEED={SEED}')


## Cell 3 — Load PubMedQA (N=1000, seed=42)

In [ ]:
from datasets import load_dataset

print('Loading PubMedQA pqa_labeled...')
ds = load_dataset('qiaojin/PubMedQA', 'pqa_labeled', split='train', trust_remote_code=True)
df_all = pd.DataFrame(ds)

label_map = {'yes': 'SUPPORTED', 'no': 'REFUTED', 'maybe': 'INCONCLUSIVE'}
df_all['gold_label'] = df_all['final_decision'].map(label_map)

def get_context(x):
    if isinstance(x, dict) and 'contexts' in x:
        return ' '.join(x['contexts'][:3])[:1200]
    return str(x)[:1200]

df_all['context_text']  = df_all['context'].apply(get_context)
df_all['question_text'] = df_all['question'].astype(str)

np.random.seed(SEED)
df = df_all.sample(n=N_SAMPLES, random_state=SEED).reset_index(drop=True)

print(f'Loaded {len(df)} samples')
print('Gold label distribution:')
print(df['gold_label'].value_counts())
print(f'\nMajority-class baseline (always predict SUPPORTED): '
      f'{(df["gold_label"]=="SUPPORTED").mean():.3f}')
print(f'Random baseline (uniform): 0.333')


In [ ]:
import os
import requests

GEMINI_API_KEY  = os.environ.get("GEMINI_API_KEY", "")   # set env var before running

url = f'https://generativelanguage.googleapis.com/v1beta/models/gemini-2.5-flash:generateContent?key={GEMINI_API_KEY}'
r = requests.post(url, json={
    'contents': [{'parts': [{'text': 'Say hello'}]}],
    'generationConfig': {'maxOutputTokens': 50}
}, timeout=30 )

print(f'Status: {r.status_code}')
print(r.text[:500])


## Cell 4 — API Functions (max_tokens=300)

In [ ]:
def call_gpt(prompt, max_tokens=300):
    """GPT-4.1-mini — max_tokens=300, temperature=0."""
    for attempt in range(3):
        try:
            r = openai_client.chat.completions.create(
                model='gpt-4.1-mini',
                messages=[
                    {'role': 'system',
                     'content': ('You are a biomedical claim verification assistant. '
                                 'Respond ONLY in the exact format requested. '
                                 'Start immediately with DECISION:')},
                    {'role': 'user', 'content': prompt}
                ],
                max_tokens=max_tokens,
                temperature=0
            )
            return r.choices[0].message.content.strip()
        except Exception as e:
            print(f'  GPT error (attempt {attempt+1}): {e}')
            time.sleep(5 * (attempt + 1))
    return ''

GEMINI_MODELS = ['gemini-2.5-flash', 'gemini-2.0-flash', 'gemini-1.5-flash']

def call_gemini(prompt, max_tokens=300):
    """Gemini-2.5-Flash with auto-fallback."""
    for model_name in GEMINI_MODELS:
        url = (f'https://generativelanguage.googleapis.com/v1beta/'
               f'models/{model_name}:generateContent?key={GEMINI_API_KEY}')
        for attempt in range(2):
            try:
                r = requests.post(url, json={
                    'contents': [{'parts': [{'text': prompt}]}],
                    'generationConfig': {'maxOutputTokens': max_tokens, 'temperature': 0},
                    'systemInstruction': {'parts': [{'text': (
                        'You are a biomedical claim verification assistant. '
                        'Respond ONLY in the exact format requested. '
                        'Start immediately with DECISION:')}]}
                }, timeout=30)
                if r.status_code == 200:
                    return r.json()['candidates'][0]['content']['parts'][0]['text'].strip()
                if r.status_code == 404:
                    break
                time.sleep(3)
            except Exception as e:
                time.sleep(3)
    return ''

def call_llama(prompt, max_tokens=300):
    """Llama-3.3-70b via TogetherAI."""
    for attempt in range(3):
        try:
            r = requests.post(
                'https://api.together.xyz/v1/chat/completions',
                headers={'Authorization': f'Bearer {TOGETHER_API_KEY}',
                         'Content-Type': 'application/json'},
                json={
                    'model': 'meta-llama/Llama-3.3-70B-Instruct-Turbo',
                    'messages': [
                        {'role': 'system',
                         'content': ('You are a biomedical claim verification assistant. '
                                     'Respond ONLY in the exact format requested. '
                                     'Start immediately with DECISION:')},
                        {'role': 'user', 'content': prompt}
                    ],
                    'max_tokens': max_tokens,
                    'temperature': 0
                },
                timeout=60
            )
            if r.status_code == 200:
                return r.json()['choices'][0]['message']['content'].strip()
            print(f'  Llama HTTP {r.status_code}: {r.text[:100]}')
            time.sleep(5)
        except Exception as e:
            print(f'  Llama error (attempt {attempt+1}): {e}')
            time.sleep(5)
    return ''

# Test all models
print('Testing all 3 models...')
test_prompt = 'Say exactly:\nDECISION: SUPPORTED\nERROR_PROBABILITY: 0.10\nACTION: COMMIT'
for name, fn in [('GPT-4.1-mini', call_gpt), ('Gemini', call_gemini), ('Llama-3.3-70b', call_llama)]:
    resp = fn(test_prompt)
    status = 'OK' if 'DECISION' in resp.upper() else 'PROBLEM'
    print(f'  {name}: {status} — {resp[:60].replace(chr(10)," | ")}')


## Cell 5 — Prompt Template

In [ ]:
PROTOCOL_A_PROMPT = """You are a biomedical claim verification assistant.

CLAIM: {claim}

PUBMED EVIDENCE: {evidence}

Complete ALL THREE tasks:

TASK 1 - DECISION: Classify the claim based on the evidence.
  Options: SUPPORTED | REFUTED | INCONCLUSIVE
  - SUPPORTED: The evidence directly supports the claim
  - REFUTED: The evidence directly contradicts the claim
  - INCONCLUSIVE: The evidence is insufficient or ambiguous

TASK 2 - MONITORING: Estimate the probability that your Task 1 decision is WRONG.
  Provide a decimal between 0.0 and 1.0.

TASK 3 - CONTROL: Based on your error probability, choose an action.
  - COMMIT: error_probability < 0.30
  - ABSTAIN: 0.30 <= error_probability < 0.60
  - SEEK_EVIDENCE: error_probability >= 0.60 and evidence is insufficient
  - REVISE: you realize a clear error in your reasoning

Respond in EXACTLY this format (no extra text before or after):
DECISION: [SUPPORTED/REFUTED/INCONCLUSIVE]
ERROR_PROBABILITY: [0.0-1.0]
ACTION: [COMMIT/ABSTAIN/SEEK_EVIDENCE/REVISE]"""

print(f'Prompt ready. Length: {len(PROTOCOL_A_PROMPT)} chars')


## Cell 6 — Parser

In [ ]:
def parse_response(text):
    """Returns (decision, error_prob, action, parse_status, notes)."""
    if not text or not text.strip():
        return 'PARSE_FAILED', None, None, 'failed', 'empty'
    t = text.upper()
    notes = []
    found = {}

    # Decision — strict then loose
    for lbl in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
        if re.search(r'DECISION\s*[:=]\s*' + lbl, t):
            found['decision'] = lbl; break
    if 'decision' not in found:
        for lbl in ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']:
            if lbl in t:
                found['decision'] = lbl
                notes.append(f'dec_fallback:{lbl}'); break

    # Error probability
    m = re.search(r'ERROR[_\s]PROB(?:ABILITY)?\s*[:=]\s*([0-9]*\.?[0-9]+)', t)
    if m:
        try: found['error_prob'] = max(0.0, min(1.0, float(m.group(1))))
        except: notes.append('prob_fail')

    # Action — strict then loose
    for act in ['SEEK_EVIDENCE', 'ABSTAIN', 'REVISE', 'COMMIT']:
        if re.search(r'ACTION\s*[:=]\s*' + act.replace('_', '[_\\s]'), t):
            found['action'] = act; break
    if 'action' not in found:
        for act in ['SEEK_EVIDENCE', 'ABSTAIN', 'REVISE', 'COMMIT']:
            if act.replace('_', ' ') in t or act in t:
                found['action'] = act
                notes.append(f'act_fallback:{act}'); break

    has_all = all(k in found for k in ['decision', 'error_prob', 'action'])
    is_clean = not any('fallback' in n for n in notes)
    status = 'full' if (has_all and is_clean) else ('partial' if found else 'failed')
    return (found.get('decision', 'PARSE_FAILED'), found.get('error_prob'),
            found.get('action'), status, '; '.join(notes) if notes else 'clean')

print('Parser ready.')


## Cell 7 — Run GPT-4.1-mini (N=1000)

**Estimated time: ~25 minutes**  
Saves checkpoint every 50 cases to `protocol_a_results_gpt.csv`


In [ ]:
MODEL_NAME = 'GPT-4.1-mini'
OUTPUT_FILE = 'protocol_a_results_gpt.csv'

# Resume from checkpoint if exists
if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_indices = set(existing['sample_idx'].tolist())
    results = existing.to_dict('records')
    print(f'Resuming from checkpoint: {len(done_indices)} cases done')
else:
    done_indices = set()
    results = []

parse_counts = {'full': 0, 'partial': 0, 'failed': 0}

for i, row in tqdm(df.iterrows(), total=len(df), desc=MODEL_NAME):
    if i in done_indices:
        continue

    prompt = PROTOCOL_A_PROMPT.format(
        claim=row['question_text'],
        evidence=row['context_text']
    )
    raw = call_gpt(prompt, max_tokens=300)
    decision, error_prob, action, parse_status, parse_notes = parse_response(raw)
    parse_counts[parse_status] += 1

    results.append({
        'model': MODEL_NAME,
        'sample_idx': i,
        'pubmed_id': str(row.get('pubid', i)),
        'gold_label': row['gold_label'],
        'decision': decision,
        'error_prob': error_prob,
        'action': action,
        'parse_status': parse_status,
        'parse_notes': parse_notes,
        'is_correct': int(decision == row['gold_label']) if decision != 'PARSE_FAILED' else 0,
        'raw_response': raw,
        'raw_length': len(raw) if raw else 0,
    })

    # Save checkpoint every BATCH_SAVE cases
    if len(results) % BATCH_SAVE == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

    time.sleep(0.3)

gpt_df = pd.DataFrame(results)
gpt_df.to_csv(OUTPUT_FILE, index=False)
print(f'\nGPT done. Saved {len(gpt_df)} rows to {OUTPUT_FILE}')
print(f'Parse: {parse_counts}')
print(f'Accuracy: {gpt_df["is_correct"].mean():.3f}')
print(f'INCONCLUSIVE rate: {(gpt_df["decision"]=="INCONCLUSIVE").mean():.1%}')


## Cell 8 — Run Gemini-2.5-Flash (N=1000)

**Estimated time: ~30 minutes**  
Free tier — no cost.


In [ ]:
MODEL_NAME = 'Gemini-2.5-Flash'
OUTPUT_FILE = 'protocol_a_results_gemini.csv'

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_indices = set(existing['sample_idx'].tolist())
    results = existing.to_dict('records')
    print(f'Resuming: {len(done_indices)} done')
else:
    done_indices = set()
    results = []

parse_counts = {'full': 0, 'partial': 0, 'failed': 0}

for i, row in tqdm(df.iterrows(), total=len(df), desc=MODEL_NAME):
    if i in done_indices:
        continue

    prompt = PROTOCOL_A_PROMPT.format(
        claim=row['question_text'],
        evidence=row['context_text']
    )
    raw = call_gemini(prompt, max_tokens=300)
    decision, error_prob, action, parse_status, parse_notes = parse_response(raw)
    parse_counts[parse_status] += 1

    results.append({
        'model': MODEL_NAME,
        'sample_idx': i,
        'pubmed_id': str(row.get('pubid', i)),
        'gold_label': row['gold_label'],
        'decision': decision,
        'error_prob': error_prob,
        'action': action,
        'parse_status': parse_status,
        'parse_notes': parse_notes,
        'is_correct': int(decision == row['gold_label']) if decision != 'PARSE_FAILED' else 0,
        'raw_response': raw,
        'raw_length': len(raw) if raw else 0,
    })

    if len(results) % BATCH_SAVE == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

    time.sleep(0.5)

gemini_df = pd.DataFrame(results)
gemini_df.to_csv(OUTPUT_FILE, index=False)
print(f'\nGemini done. Saved {len(gemini_df)} rows to {OUTPUT_FILE}')
print(f'Parse: {parse_counts}')
print(f'Accuracy: {gemini_df["is_correct"].mean():.3f}')
print(f'INCONCLUSIVE rate: {(gemini_df["decision"]=="INCONCLUSIVE").mean():.1%}')


## Cell 9 — Run Llama-3.3-70b (N=1000)

**Estimated time: ~30 minutes**  
Via TogetherAI API.


In [ ]:
MODEL_NAME = 'Llama-3.3-70b'
OUTPUT_FILE = 'protocol_a_results_llama.csv'

if os.path.exists(OUTPUT_FILE):
    existing = pd.read_csv(OUTPUT_FILE)
    done_indices = set(existing['sample_idx'].tolist())
    results = existing.to_dict('records')
    print(f'Resuming: {len(done_indices)} done')
else:
    done_indices = set()
    results = []

parse_counts = {'full': 0, 'partial': 0, 'failed': 0}

for i, row in tqdm(df.iterrows(), total=len(df), desc=MODEL_NAME):
    if i in done_indices:
        continue

    prompt = PROTOCOL_A_PROMPT.format(
        claim=row['question_text'],
        evidence=row['context_text']
    )
    raw = call_llama(prompt, max_tokens=300)
    decision, error_prob, action, parse_status, parse_notes = parse_response(raw)
    parse_counts[parse_status] += 1

    results.append({
        'model': MODEL_NAME,
        'sample_idx': i,
        'pubmed_id': str(row.get('pubid', i)),
        'gold_label': row['gold_label'],
        'decision': decision,
        'error_prob': error_prob,
        'action': action,
        'parse_status': parse_status,
        'parse_notes': parse_notes,
        'is_correct': int(decision == row['gold_label']) if decision != 'PARSE_FAILED' else 0,
        'raw_response': raw,
        'raw_length': len(raw) if raw else 0,
    })

    if len(results) % BATCH_SAVE == 0:
        pd.DataFrame(results).to_csv(OUTPUT_FILE, index=False)

    time.sleep(0.3)

llama_df = pd.DataFrame(results)
llama_df.to_csv(OUTPUT_FILE, index=False)
print(f'\nLlama done. Saved {len(llama_df)} rows to {OUTPUT_FILE}')
print(f'Parse: {parse_counts}')
print(f'Accuracy: {llama_df["is_correct"].mean():.3f}')
print(f'INCONCLUSIVE rate: {(llama_df["decision"]=="INCONCLUSIVE").mean():.1%}')


## Cell 10 — Combine Results & Compute Baselines

In [ ]:
# Load all three result files
gpt_df   = pd.read_csv('protocol_a_results_gpt.csv')
gemini_df = pd.read_csv('protocol_a_results_gemini.csv')
llama_df  = pd.read_csv('protocol_a_results_llama.csv')

all_df = pd.concat([gpt_df, gemini_df, llama_df], ignore_index=True)
all_df.to_csv('protocol_a_results_ALL.csv', index=False)
print(f'Combined: {len(all_df)} rows saved to protocol_a_results_ALL.csv')

# Compute baselines on the gold labels
gold = gpt_df['gold_label']  # same for all models
majority_class = gold.value_counts().index[0]
majority_acc = (gold == majority_class).mean()
random_acc = 1.0 / gold.nunique()

print(f'\nBaselines (N={len(gold)}):')
print(f'  Majority-class ({majority_class}): {majority_acc:.3f}')
print(f'  Random (uniform):                  {random_acc:.3f}')
print(f'  Random (stratified):               {sum((gold==l).mean()**2 for l in gold.unique()):.3f}')

print(f'\nGold label distribution:')
print(gold.value_counts())


## Cell 11 — Full Summary Table (Paper Table 1)

In [ ]:
all_df = pd.read_csv('protocol_a_results_ALL.csv')
gold = pd.read_csv('protocol_a_results_gpt.csv')['gold_label']

majority_acc = (gold == gold.value_counts().index[0]).mean()
random_acc = 1.0 / gold.nunique()

print('='*75)
print('PROTOCOL A RESULTS — CLEAN (N=1000, max_tokens=300)')
print('='*75)

summary_rows = []
for model in ['GPT-4.1-mini', 'Gemini-2.5-Flash', 'Llama-3.3-70b']:
    sub = all_df[all_df['model'] == model]
    if len(sub) == 0:
        print(f'{model}: no data')
        continue

    valid = sub[sub['decision'] != 'PARSE_FAILED']
    acc = (valid['decision'] == valid['gold_label']).mean()
    inc_rate = (sub['decision'] == 'INCONCLUSIVE').mean()
    parse_fail = (sub['parse_status'] == 'failed').mean()
    full_parse = (sub['parse_status'] == 'full').mean()
    commit_rate = (sub['action'] == 'COMMIT').mean()
    abstain_rate = (sub['action'] == 'ABSTAIN').mean()
    seek_rate = (sub['action'] == 'SEEK_EVIDENCE').mean()

    summary_rows.append({
        'Model': model,
        'N': len(sub),
        'Accuracy': round(acc, 3),
        'INCONCLUSIVE%': f'{inc_rate:.1%}',
        'Parse_Full%': f'{full_parse:.1%}',
        'Parse_Fail%': f'{parse_fail:.1%}',
        'COMMIT%': f'{commit_rate:.1%}',
        'ABSTAIN%': f'{abstain_rate:.1%}',
        'SEEK%': f'{seek_rate:.1%}',
    })

# Add baselines
summary_rows.append({'Model': 'Majority-class baseline', 'N': len(gold),
                     'Accuracy': round(majority_acc, 3), 'INCONCLUSIVE%': '0%',
                     'Parse_Full%': '100%', 'Parse_Fail%': '0%',
                     'COMMIT%': '-', 'ABSTAIN%': '-', 'SEEK%': '-'})
summary_rows.append({'Model': 'Random baseline (uniform)', 'N': len(gold),
                     'Accuracy': round(random_acc, 3), 'INCONCLUSIVE%': '33%',
                     'Parse_Full%': '100%', 'Parse_Fail%': '0%',
                     'COMMIT%': '-', 'ABSTAIN%': '-', 'SEEK%': '-'})

summary_df = pd.DataFrame(summary_rows)
print(summary_df.to_string(index=False))

# Save summary
summary_text = summary_df.to_string(index=False)
with open('protocol_a_summary.txt', 'w') as f:
    f.write('PROTOCOL A RESULTS — N=1000, max_tokens=300\n')
    f.write('='*75 + '\n')
    f.write(summary_text)
    f.write('\n\nGold label distribution:\n')
    f.write(gold.value_counts().to_string())
print('\nSaved: protocol_a_summary.txt')


## Cell 12 — Confusion Matrices (3 Models)

In [ ]:
all_df = pd.read_csv('protocol_a_results_ALL.csv')
models = ['GPT-4.1-mini', 'Gemini-2.5-Flash', 'Llama-3.3-70b']
gold_labels = ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE']
pred_labels = ['SUPPORTED', 'REFUTED', 'INCONCLUSIVE', 'PARSE_FAILED']

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

for ax, model in zip(axes, models):
    sub = all_df[all_df['model'] == model]
    if len(sub) == 0:
        ax.set_title(f'{model}\nNo data')
        continue

    cm = pd.crosstab(
        sub['gold_label'], sub['decision'],
        rownames=['Gold'], colnames=['Predicted']
    ).reindex(index=gold_labels, columns=pred_labels, fill_value=0)

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                linewidths=0.5, linecolor='gray')

    acc = sub['is_correct'].mean()
    inc = (sub['decision'] == 'INCONCLUSIVE').mean()
    ax.set_title(f'{model}\nAcc={acc:.3f}  |  INC%={inc:.1%}', fontsize=11)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('Gold')

plt.suptitle('Protocol A Confusion Matrices (N=1000, max_tokens=300)',
             fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig('protocol_a_confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved: protocol_a_confusion_matrices.png')


## Cell 13 — INCONCLUSIVE Rate Analysis

In [ ]:
all_df = pd.read_csv('protocol_a_results_ALL.csv')

print('='*65)
print('INCONCLUSIVE RATE ANALYSIS')
print('='*65)
print(f'Gold label INCONCLUSIVE base rate: '
      f'{(all_df[all_df["model"]=="GPT-4.1-mini"]["gold_label"]=="INCONCLUSIVE").mean():.1%}')
print()

for model in ['GPT-4.1-mini', 'Gemini-2.5-Flash', 'Llama-3.3-70b']:
    sub = all_df[all_df['model'] == model]
    if len(sub) == 0: continue

    inc_rate = (sub['decision'] == 'INCONCLUSIVE').mean()
    # Among INCONCLUSIVE predictions, how many are correct?
    inc_preds = sub[sub['decision'] == 'INCONCLUSIVE']
    inc_precision = (inc_preds['gold_label'] == 'INCONCLUSIVE').mean() if len(inc_preds) > 0 else 0

    # Among gold INCONCLUSIVE, how many did model find?
    gold_inc = sub[sub['gold_label'] == 'INCONCLUSIVE']
    inc_recall = (gold_inc['decision'] == 'INCONCLUSIVE').mean() if len(gold_inc) > 0 else 0

    print(f'{model}:')
    print(f'  Predicted INCONCLUSIVE: {inc_rate:.1%} ({int(inc_rate*len(sub))} cases)')
    print(f'  INCONCLUSIVE precision: {inc_precision:.1%} (of predicted INC, how many are truly INC)')
    print(f'  INCONCLUSIVE recall:    {inc_recall:.1%} (of true INC, how many did model find)')
    print()

print('INTERPRETATION:')
print('  If predicted INC% >> gold INC%: model is over-hedging')
print('  This is the "commitment avoidance" finding for the paper')


## Cell 14 — Final Paper Numbers

**Run this last — copy these numbers directly into the paper.**


In [ ]:
all_df = pd.read_csv('protocol_a_results_ALL.csv')
gold = all_df[all_df['model']=='GPT-4.1-mini']['gold_label']
majority_acc = (gold == gold.value_counts().index[0]).mean()

print('='*70)
print('FINAL NUMBERS FOR PAPER — Protocol A (N=1000, max_tokens=300)')
print('='*70)
print()
print(f'Gold label distribution:')
for lbl, cnt in gold.value_counts().items():
    print(f'  {lbl}: {cnt} ({cnt/len(gold):.1%})')
print()
print(f'Baselines:')
print(f'  Majority-class (always SUPPORTED): {majority_acc:.3f}')
print(f'  Random (uniform 3-class):          {1/3:.3f}')
print()

for model in ['GPT-4.1-mini', 'Gemini-2.5-Flash', 'Llama-3.3-70b']:
    sub = all_df[all_df['model'] == model]
    if len(sub) == 0: continue
    valid = sub[sub['decision'] != 'PARSE_FAILED']
    acc = (valid['decision'] == valid['gold_label']).mean()
    inc = (sub['decision'] == 'INCONCLUSIVE').mean()
    parse_fail = (sub['parse_status'] == 'failed').mean()
    print(f'{model}:')
    print(f'  Accuracy:          {acc:.3f}')
    print(f'  INCONCLUSIVE rate: {inc:.1%}')
    print(f'  Parse fail rate:   {parse_fail:.1%}')
    print(f'  N valid:           {len(valid)}')
    print()

print('=> Copy these numbers into Table 1 of the paper.')
print('=> Add majority-class and random baselines as rows in the table.')
print('=> Report parse fail rate in footnote or limitations.')
